# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOmerSiddiqui/myInternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
# Setup — run this FIRST
import os, sys, subprocess
import pandas as pd
import duckdb

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "Set HF_TOKEN in Colab Secrets (key icon) or as environment variable"

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB ready. Token loaded.")
print("con and rel are now defined.")

DuckDB ready. Token loaded.
con and rel are now defined.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My lane:** Refresh / Content Opportunity Scoring

**Unit of analysis:**  
One row = one content page (content_hash_id) aggregated over a time window.  
In the daily fact table the raw grain is one row = one (report_date × client × content).  
For scoring I will roll it up to one row per content item.

**Time window I will use for development:**  
Month = 2026-03 (a mid-panel month).  
I deliberately avoid the final month (2026-06 / `_sample`) so I do not develop labels inside the natural outcome window.

**Tables I will use:**  
- `fact_content_daily_performance` (partitioned by month) — main signals  
- `dim_content` — content metadata (when needed)  
- `dim_clients` — history start dates

**What I will rank / predict (proxy for now):**  
A decline / opportunity score based on observed performance in the chosen window (later I can define a true forward-looking label).

**One thing I deliberately exclude:**  
Any product decision flags (health_score, priority_score, action_type) — they are not in the release and would be circular if I rebuilt them.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature candidates (knowable before the decision):**  
- impressions, clicks, avg position (GSC)  
- sessions, engagement / scroll signals (GA4, only where ga4_data_available IS TRUE)  
- content age / days since last update (from dim_content)  
- word count / content type (metadata)

**Label / proxy:**  
- A decline or opportunity label I define from a *later* window (or the starter-style trend proxy while learning).  
  Never use the same window’s trend as both feature and label.

**Context only (never features):**  
- content_hash_id, client_hash_id, url_hash_id, keyword_hash_id  
- report_date / month partitions

**Excluded (and why):**  
- Raw client names, domains, URLs, titles, keywords → not present and must never be reconstructed  
- Product scores / action flags → circular and not shipped  
- Future-window metrics when building features → leakage

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries on month=2026-03:

1. Grain check — does one row really mean one (date × client × content)?
2. Row count + date span of my slice
3. Availability filter using `IS TRUE` (how many rows survive)

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ----- Query 1: Grain check -----
print("=== 1. Grain check (should return 0 rows) ===")
q1 = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print(q1)
print("If empty → grain holds: one row = one (date × client × content)\n")

# ----- Query 2: Row count + date span -----
print("=== 2. Row count and date span for month=2026-03 ===")
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
""").df()
display(q2)

# ----- Query 3: Availability filter with IS TRUE -----
print("=== 3. Rows where ga4_data_available IS TRUE ===")
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_ga4_true
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
""").df()
display(q3)

=== 1. Grain check (should return 0 rows) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
If empty → grain holds: one row = one (date × client × content)

=== 2. Row count and date span for month=2026-03 ===


,row_count,unique_pages,unique_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


=== 3. Rows where ga4_data_available IS TRUE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_true_rows,pct_ga4_true
0,9841378,413966,4.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Five features (with “available when?” line)

I build a small content-level feature frame from March 2026.

| Feature | Why knowable at decision time |
|---------|-------------------------------|
| total_impressions | Sum of past impressions in the feature window only |
| avg_position | Average position observed in the same past window |
| total_clicks | Clicks already recorded before the decision |
| days_with_impressions | How many days the page was visible in the past window |
| total_sessions | Sessions only where ga4_data_available IS TRUE |

### The trap (deliberate leakage)
I will intentionally add a label-derived column, watch a quick score jump, then remove it.

### Named limitation of this slice
- History is an **unbalanced panel** — different clients start at different dates.  
- Early rows can be GSC-only (`ga4_data_available` is not TRUE).  
- Using the final month for label development would leak the outcome window.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ----- Build a 5-feature frame (content-level, March 2026) -----
print("=== Building 5 honest features ===")

features = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position,
    SUM(gsc_clicks) AS total_clicks,
    COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
    SUM(COALESCE(sessions_organic, 0) + COALESCE(sessions_direct, 0) +
        COALESCE(sessions_referral, 0) + COALESCE(sessions_social, 0) +
        COALESCE(sessions_paid, 0) + COALESCE(sessions_ai, 0)
    ) FILTER (WHERE ga4_data_available IS TRUE) AS total_sessions
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
GROUP BY 1, 2
HAVING SUM(gsc_impressions) > 0
""").df()

print("Feature frame shape:", features.shape)
display(features.head())

# Simple proxy label for the trap demo
features["proxy_label"] = (
    (features["total_impressions"] > features["total_impressions"].median()) &
    (features["avg_position"].fillna(999) > 10)
).astype(int)

print("\nProxy label rate:", round(features["proxy_label"].mean(), 3))

# ----- THE TRAP: deliberately leak the label into a feature -----
print("\n=== DELIBERATE LEAK EXPERIMENT ===")
features_leaky = features.copy()
features_leaky["leaky_feature"] = features_leaky["proxy_label"]   # THIS IS THE TRAP

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X_honest = features[["total_impressions", "avg_position", "total_clicks",
                     "days_with_impressions", "total_sessions"]].fillna(0)
X_leaky  = features_leaky[["total_impressions", "avg_position", "total_clicks",
                           "days_with_impressions", "total_sessions", "leaky_feature"]].fillna(0)
y = features["proxy_label"]

honest_score = cross_val_score(LogisticRegression(max_iter=500), X_honest, y, cv=3, scoring="roc_auc").mean()
leaky_score  = cross_val_score(LogisticRegression(max_iter=500), X_leaky,  y, cv=3, scoring="roc_auc").mean()

print(f"Honest 5-feature ROC-AUC : {honest_score:.3f}")
print(f"With leaky column ROC-AUC: {leaky_score:.3f}  ← jumps toward perfect")
print("→ Now we DELETE the leaky column and keep only the honest number.")

del features_leaky
print("\nLeak removed. We keep only the honest feature set.")

=== Building 5 honest features ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)


,content_hash_id,client_hash_id,total_impressions,avg_position,total_clicks,days_with_impressions,total_sessions
0,content_67741cce996cfafa,client_62f4a7e64f5e0096,46.0,5.942308,1.0,16,NaN
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,5.908100,1.0,31,NaN
2,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3108.0,6.969536,0.0,30,NaN
3,content_275b6f7f733016d4,client_62f4a7e64f5e0096,810.0,4.866123,1.0,29,NaN
4,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,134.0,5.012831,0.0,26,NaN



Proxy label rate: 0.216

=== DELIBERATE LEAK EXPERIMENT ===
Honest 5-feature ROC-AUC : 0.894
With leaky column ROC-AUC: 1.000  ← jumps toward perfect
→ Now we DELETE the leaky column and keep only the honest number.

Leak removed. We keep only the honest feature set.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.